# Hands-on: Train an ensemble AIFS (AIFS-CRPS) with Anemoi

In the last session we trained a deterministic model with a mean squared error loss. In this session we turn it into an **ensemble** model like the AIFS-ENS, trained with the **CRPS** (Continuous Ranked Probability Score).

**Learning objectives.** By the end of this session you will be able to:
- explain why a model trained with the MSE gives smooth forecasts, and how the CRPS leads to an ensemble
- name the parts of the config that change for CRPS training, and what each does
- train an ensemble model and look at its ensemble members

**Resources**
- [anemoi-training: CRPS-based training](https://anemoi.readthedocs.io/projects/training/en/latest/user-guide/kcrps-set-up.html)
- [Lang et al. 2024, AIFS-CRPS: ensemble forecasting using a model trained with a loss function based on the CRPS](https://arxiv.org/abs/2412.15832)

## Background: from MSE to CRPS

A deterministic model trained with the MSE learns to predict the **mean** of all futures that are possible given the input. When the future is uncertain, this mean is smooth: small-scale features that the model cannot predict are averaged out. The longer the forecast, the smoother it gets.

An ensemble model instead produces several **members**, each a possible future. To train it we need a loss that rewards an ensemble whose members are spread like the possible futures. The CRPS is such a loss. For an ensemble $x_1, \dots, x_M$ and the truth $y$, at each grid point and for each variable:

$$\text{CRPS} = \frac{1}{M}\sum_{i=1}^{M} |x_i - y| \; - \; \frac{1}{2M^2}\sum_{i=1}^{M}\sum_{j=1}^{M} |x_i - x_j|$$

- the first term rewards members **close to the truth** (with one member it is the absolute error)
- the second term rewards members that are **different from each other**, which stops them from all predicting the mean

The *fair* CRPS (fCRPS) divides the second term by $2M(M-1)$ instead of $2M^2$. With it, the expected score does not depend on the number of members, so a model trained with only 2 members per sample learns the spread that a large ensemble should have. The AIFS uses the *almost fair* CRPS, a mix of both controlled by $\alpha$:

$$\text{afCRPS}_\alpha = \alpha \, \text{fCRPS} + (1-\alpha)\, \text{CRPS}$$

<img src="_resources/aifs-crps_sketch.png" alt="Sketch of the AIFS-CRPS training" width="900">

A quick experiment: the truth is drawn from a normal distribution with standard deviation 1, and we score ensembles drawn from normal distributions with different spreads. Which spread gets the best (lowest) score?

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np


def crps(members, truth, fair=True):
    """CRPS of an ensemble (members along axis 0) for each truth value."""
    m = members.shape[0]
    skill = np.abs(members - truth).mean(axis=0)
    spread = np.abs(members[:, None] - members[None, :]).sum(axis=(0, 1))
    return skill - spread / (2 * m * (m - 1) if fair else 2 * m**2)


rng = np.random.default_rng(0)
truth = rng.normal(size=100_000)
spreads = np.linspace(0, 2, 41)
for m, fair in [(2, True), (2, False), (8, False)]:
    scores = [crps(rng.normal(scale=s, size=(m, truth.size)), truth, fair).mean() for s in spreads]
    plt.plot(spreads, scores, label=f"{'fair' if fair else 'standard'} CRPS, {m} members")
plt.axvline(1, color="k", ls=":", label="spread of the truth")
plt.xlabel("standard deviation of the ensemble")
plt.ylabel("mean score")
plt.legend()

The fair CRPS is lowest when the ensemble has the spread of the truth, even with only 2 members. The standard CRPS with few members rewards ensembles that are too narrow.

## 1. What changes in the config

We use the deterministic config of the last session and the ensemble config `configs/crps_o48.yaml`, and let the computer list every setting that differs (leaving out the plots and logging).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf

from helpers import config_diff, load_config, run_training

# The toy dataset and where the trainings write their output. On the course server nothing needs to change here.
DATASET = "/datasets/aifs-ea-an-oper-0001-mars-o48-2010-2022-6h-v1-toy.zarr"
OUTPUT_ROOT = Path.home() / "anemoi-output"

# Settings passed to every training and config in this notebook (Hydra "overrides", see below)
OVERRIDES = [f"system.input.dataset={DATASET}", f"system.output.root={OUTPUT_ROOT}/"]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, training will be very slow")

In [ ]:
print(Path("configs/crps_o48.yaml").read_text())

In [ ]:
deterministic = load_config("deterministic_o48", OVERRIDES)
ensemble = load_config("crps_o48", OVERRIDES)

pd.set_option("display.max_colwidth", 70)
diff = pd.DataFrame(config_diff(deterministic, ensemble), columns=["setting", "deterministic", "ensemble"])
diff

These differences fall into three groups.

### 1.1 The model: noise, and layers that listen to it
| | Deterministic | Ensemble |
|---|---|---|
| model class | `AnemoiModelEncProcDec` | `AnemoiEnsModelEncProcDec` |
| noise | none | `noise_injector`: `NoiseConditioning` |
| processor layer norm | `LayerNorm` | `ConditionalLayerNorm` |

To produce different members from the same input, the model needs randomness. At every forecast step, the **noise injector** draws a random vector for each member (`noise_channels_dim` = 4 numbers) and embeds it with a small MLP. Every layer norm in the processor is replaced by a **conditional layer norm**, whose scale and shift are computed from this noise embedding. So the noise changes the computation in every processor layer, and each member takes a different path through the same weights.

With `zero_init: True` the conditional layer norms start out ignoring the noise, so at the start of the training all members are equal, and the model learns how much the noise should matter.

In [ ]:
print(OmegaConf.to_yaml(ensemble.model.noise_injector))
print(OmegaConf.to_yaml(ensemble.model.processor.layer_kernels.LayerNorm))

### 1.2 The training task: several members per sample
| | Deterministic | Ensemble |
|---|---|---|
| task | `GraphForecaster` | `GraphEnsForecaster` |
| strategy | `DDPGroupStrategy` | `DDPEnsGroupStrategy` |
| members | - | `ensemble_size_per_device` |

The `GraphEnsForecaster` runs the model `ensemble_size_per_device` times on each sample, each time with different noise, and computes the loss on all members together. With more GPUs, the members of one ensemble can be spread over several GPUs (`system.hardware.num_gpus_per_ensemble`), and the `DDPEnsGroupStrategy` organises which GPUs work on the same ensemble. On our single GPU:

ensemble size = `ensemble_size_per_device` × `num_gpus_per_ensemble` = 2 × 1 = **2 members**

In [ ]:
print("members per GPU:", ensemble.training.ensemble_size_per_device)
print("GPUs per ensemble:", ensemble.system.hardware.num_gpus_per_ensemble)

### 1.3 The loss: CRPS, on several scales
The training loss is the almost fair CRPS with $\alpha = 1$ (fully fair), wrapped in a `MultiscaleLossWrapper`. This wrapper computes the CRPS not only on the fields themselves, but also on 3 **smoothed** versions of them (Gaussian smoothing over 4, 2 and 1 times the grid spacing). The final loss is the weighted sum (weights 1, 1, 1 and 4 for the unsmoothed fields).

Why? The CRPS at each grid point does not care whether the members have realistic spatial patterns. Scoring the smoothed fields as well rewards members whose large-scale features are right, not only their values at each point. The smoothing is a sparse matrix multiplication, with matrices computed in advance (`loss_matrices/`):

In [ ]:
import scipy.sparse
from anemoi.datasets import open_dataset

ds = open_dataset(DATASET)
field = ds[0, ds.name_to_index["tcw"], 0]  # total column water: lots of small-scale structure

loss = ensemble.training.training_loss.datasets.data
smoothed = {"no smoothing": field}
for name in reversed(loss.loss_matrices[:-1]):  # from the least to the most smoothing
    matrix = scipy.sparse.load_npz(Path(ensemble.system.input.loss_matrices_path) / name)
    smoothed["smoothed over " + name.split("_d=")[1].removesuffix(".npz") + " the grid spacing"] = matrix @ field

import cartopy.crs as ccrs
lons = (ds.longitudes + 180) % 360 - 180
fig, axes = plt.subplots(1, 4, figsize=(20, 4), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, (title, values) in zip(axes, smoothed.items()):
    ax.scatter(lons, ds.latitudes, c=values, s=25, cmap="Blues", vmin=0, vmax=40, transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.set_extent([-70, 30, 20, 70])
    ax.set_title(f"tcw, {title}")

## 2. Train

The training takes about **5-10 minutes** on the course GPU. As before, you can also run it in a terminal:

```bash
export ANEMOI_BASE_SEED=42
anemoi-training train --config-path=$PWD/configs --config-name=crps_o48
```

In [ ]:
run_training("crps_o48", OVERRIDES)

## 3. Follow the training

In [ ]:
client = mlflow.tracking.MlflowClient(tracking_uri=str(OUTPUT_ROOT / "logs" / "mlflow"))


def latest_run(run_name):
    """The most recent MLflow run with this name (the run_name set in the config)."""
    experiment = client.get_experiment_by_name("training-course")
    runs = client.search_runs([experiment.experiment_id], f"tags.mlflow.runName = '{run_name}'",
                              order_by=["attributes.start_time DESC"], max_results=1)
    if not runs:
        raise ValueError(f"No run called {run_name!r} yet")
    return runs[0]


def metric_history(run, metric):
    history = client.get_metric_history(run.info.run_id, metric)
    return pd.Series([m.value for m in history], index=[m.step for m in history], name=metric).sort_index()


def plot_metrics(runs, metrics, ylabel="loss"):
    """Plot the history of some metrics of one or more runs, one panel per metric."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4), squeeze=False)
    for ax, metric in zip(axes[0], metrics):
        for run in runs:
            values = metric_history(run, metric)
            ax.plot(values.index, values.values, ".-" if len(values) < 30 else "-", label=run.info.run_name)
        ax.set(title=metric, xlabel="training step", ylabel=ylabel, yscale="log")
        ax.legend()
    plt.tight_layout()

In [ ]:
run = latest_run("crps-o48")
print(f"run {run.info.run_name} ({run.info.run_id}), status {run.info.status}")
print("metrics:", sorted(run.data.metrics))

In [ ]:
plot_metrics([run], ["train_multi_dataset_loss_epoch", "val_multi_dataset_loss_epoch"])

# The fair CRPS of single variables, in physical units, as a validation metric
crps_metrics = {k: v for k, v in run.data.metrics.items() if k.startswith("val_data_fkcrps_metric/")}
pd.Series(crps_metrics, name="fair CRPS").sort_index()

**What to look for**
- The loss falls like the MSE did. It is the sum of the CRPS on the 4 scales, weighted 1, 1, 1 and 4, in normalised units, so it cannot be compared directly with the MSE of the last session.
- In the plot of the ensemble members below: do the members differ? Where? After so few training steps, the model has not yet learned much spread.

In [ ]:
# The plots the training made during validation (diagnostics.plot in the config)
plot_dir = OUTPUT_ROOT / "plots" / run.info.run_id
plot_files = sorted(plot_dir.rglob("*sample*"), key=lambda f: f.stat().st_mtime)
print(f"{len(plot_files)} plots in {plot_dir}")

from IPython.display import Image, display
display(Image(filename=plot_files[-1]))  # the last one

## 4. Tasks

### Task 1: More members

Retrain with 4 instead of 2 members per GPU, with its own run name:

```python
run_training("crps_o48", OVERRIDES + ["training.ensemble_size_per_device=4", "diagnostics.log.mlflow.run_name=crps-o48-4members"])
```

**What to look for**
- **Cost:** how much longer does a step take? How much more GPU memory is used (`nvidia-smi` in a terminal)? Every member is a full forward and backward pass of the model.
- **Loss:** compare the validation CRPS of both runs with `plot_metrics`. The fair CRPS does not depend on the ensemble size, so the curves can be compared directly. Does the extra cost buy a lower CRPS after the same number of steps?
- **Limits:** what limits the ensemble size you can train with on one GPU?

In [ ]:
# Task 1: your training and comparison

### Task 2: Is the multi-scale loss worth it?

Train with the CRPS on the unsmoothed fields only:

```python
run_training("crps_o48", OVERRIDES + [
    "training.training_loss.datasets.data.loss_matrices=[null]",
    "training.training_loss.datasets.data.weights=[1.0]",
    "diagnostics.log.mlflow.run_name=crps-o48-single-scale",
])
```

**What to look for**
- Compare the validation `fkcrps` metrics (`val_data_fkcrps_metric/...`) of both runs. It is the CRPS on the unsmoothed fields, so it is the same measure for both. Which run is better on it, and why might that be?
- Look at the ensemble plots of both runs. Do the members of one run look more realistic (less noisy at small scales)?

In [ ]:
# Task 2: your training and comparison

### Task 3 (optional): Run your ensemble model

Open `../3-run-AIFS/inference-aifs-ens-o48-cpu.ipynb` and, after the download cell, point `ckpt_file` to your inference checkpoint:

```python
ckpt_file = Path.home() / "anemoi-output/checkpoint/<run id>/inference-last.ckpt"
```

Run two forecasts from the same initial conditions. Do they differ? How does the difference grow with lead time, compared with the model provided in that notebook?

In [ ]:
print(OUTPUT_ROOT / "checkpoint" / run.info.run_id / "inference-last.ckpt")